# Sub-1B document-VLM comparison on a T4 GPU

This notebook is intentionally **thin**: it only *clones the repo, installs it, and runs the repo's scripts*. All logic lives in `scripts/` and `src/docvlm_eval/` so results are reproducible and the notebook stays a runner.

It runs every sub-1B model (InternVL2/2.5/3-1B, SmolVLM-256M/500M, SmolDocling, LLaVA-OV-0.5B, GOT-OCR2, Florence-2 base/large, H2OVL-0.8B, Ovis2-1B) **plus PaddleOCR-VL 1.0/1.5/1.6**, and measures **score + inference time + CPU/GPU memory** (recorded by the model wrapper) for each.


## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2. Clone my repo + install (editable)

In [ ]:
%cd /content
![ -d OCR ] || git clone https://github.com/SangbumChoi/OCR.git
%cd /content/OCR
!git checkout claude/new-session-w79q0i && git pull --ff-only
!pip -q install -e .

## 3. Run the full comparison (repo script — two transformers passes, all measured)
`run_full_comparison.sh` installs transformers 4.49 for the chat VLMs, then 4.57 for PaddleOCR-VL, runs each on the capability + spatial/context probes, and aggregates.

In [ ]:
!DEVICE=cuda bash scripts/run_full_comparison.sh

## 4. Results — scores + efficiency (time & memory)

In [ ]:
print(open('results/matrix_capability.md').read())

## 5. Spatial / context shortcut-robust signals

In [ ]:
print(open('results/matrix_probe.md').read())
!python scripts/analyze_probe_signals.py --probe probe

## 6. PaddleOCR-VL 1.0 vs 1.5 vs 1.6 (per-sample)

In [ ]:
!for m in paddleocr-vl paddleocr-vl-1.5 paddleocr-vl-1.6; do echo "== $m =="; cat results/$m/capability/summary.json 2>/dev/null | python3 -m json.tool | grep -E 'score|latency|peak|load' ; done

## 7. Download all results

In [ ]:
!zip -qr /content/docvlm_results.zip results
from google.colab import files; files.download('/content/docvlm_results.zip')